In [28]:
# -----------------------------
# Imports
# -----------------------------
import os
import re
import json
import sys
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI

# Add the parent folder (week6) to Python path
sys.path.append(os.path.abspath(os.path.join("..")))  

# Now import normally
from pricer.items import Item
from pricer.evaluator import evaluate

In [29]:
# -----------------------------
# Config
# -----------------------------
LITE_MODE = True  # True if your system is low on memory/compute
TRAIN_SIZE = 500   # number of examples to fine-tune on
VAL_SIZE = 100
MAX_PRICE = 1000   # max clip for predictions
MIN_PRICE = 1      # min clip for predictions


In [30]:
# -----------------------------
# Load Environment
# -----------------------------
load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

openai = OpenAI(timeout=60)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [31]:
# -----------------------------
# Load Dataset
# -----------------------------
username = "ed-donner"
dataset_name = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset_name)
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")


README.md:   0%|          | 0.00/735 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.07M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/304k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/304k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


In [32]:
def clean_text(text: str) -> str:
    if not text:
        return ""
    # Remove extra spaces, newlines
    text = re.sub(r"\s+", " ", text)
    # Remove weird characters except common punctuation
    text = re.sub(r"[^\w\s.,-]", "", text)
    return text.strip()

In [33]:
def build_product_text(item: Item) -> str:
    parts = []
    if item.title:
        parts.append(f"Title: {clean_text(item.title)}")
    if item.category:
        parts.append(f"Category: {clean_text(item.category.replace('_', ' '))}")
    if item.weight:
        parts.append(f"Weight: {item.weight} lbs")
    if item.summary:
        parts.append(f"Summary: {clean_text(item.summary)}")
    if item.full:
        parts.append(f"Description: {clean_text(item.full[:500])}")  # truncate long full text
    return "\n".join(parts)

In [34]:
def messages_for(item: Item):
    product_text = build_product_text(item)
    message = f"""
You are a retail pricing expert.

Estimate the typical retail price of this product in USD.

Respond only with a single number (no $ sign, no text).

{product_text}
"""
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"{round(item.price)}"}  # rounded price
    ]

In [35]:
def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str + '}\n'
    return result.strip()

    #this will create a jsonl file in your current directory

def write_jsonl(items, filename):
    with open(filename, "w") as f:
        f.write(make_jsonl(items))

In [36]:
# -----------------------------
# Prepare Training/Validation
# -----------------------------
fine_tune_train = train[:TRAIN_SIZE]
fine_tune_val = val[:VAL_SIZE]

os.makedirs("jsonl", exist_ok=True)
write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")
write_jsonl(fine_tune_val, "jsonl/fine_tune_validation.jsonl")

In [37]:
# Upload Files for Fine-Tuning
# -----------------------------
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    val_file = openai.files.create(file=f, purpose="fine-tune")

In [39]:
fine_tuning_job = openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model="gpt-4.1-mini-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="pricer_improved"
)

job_id = fine_tuning_job.id
print("Fine-tuning job ID:", job_id)

Fine-tuning job ID: ftjob-oEWw19w8Vxwra8CV3QSKGchX


In [16]:

fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model
# fine_tuned_model_name = None

In [17]:
fine_tuned_model_name

'ft:gpt-4.1-nano-2025-04-14:personal:pricer-improved:DGmI1no2'

In [19]:
# -----------------------------
# Safe Inference
# -----------------------------
def test_messages_for(item: Item):
    return [{"role": "user", "content": build_product_text(item)}]

def gpt_fine_tuned_predict(item: Item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7
    )
    raw_value = response.choices[0].message.content
    # Clip predictions to safe range
    try:
        value = float(re.search(r"[-+]?\d*\.\d+|\d+", raw_value).group())
    except:
        value = MIN_PRICE
    return max(MIN_PRICE, min(value, MAX_PRICE))

In [ ]:
evaluate(gpt_fine_tuned_predict, test)